# Finance LLM Eval

## Read Test Dataset

In [ ]:
import pandas as pd
from transformers import AutoTokenizer

from pathlib import Path

# for .py in scripts
# PROJECT_ROOT  = Path(__file__).resolve().parent.parent

# for notebooks folder
PROJECT_ROOT = Path.cwd().parent
project_root = str(PROJECT_ROOT)

seed = 42
trained_weights = "mistralai/Mistral-7B-Instruct-v0.2"

In [ ]:
import os

# Should configure this
project_name = "finance-llm"

# Paths to save the model
output_dir = os.path.join(project_root, "outputs/")
checkpoints_path = os.path.join(output_dir, project_name + "-checkpoints")
adapter_path = os.path.join(output_dir, project_name + "-adapter")

tokenizer = AutoTokenizer.from_pretrained(adapter_path)

In [ ]:
dataset_dir = PROJECT_ROOT / "data" / "fiqa"
test_csv = "test_df.csv"
test_df = pd.read_csv(str(dataset_dir / test_csv))

test_df.head()

In [ ]:
from alpha_crunch.finance_llm.dataset import build_hf_dataset

test_dataset  = build_hf_dataset(test_df,  tokenizer, add_answer=False)
test_dataset["chat"][0]

In [ ]:
from alpha_crunch.finance_llm.dataset import tokenize_for_eval

test_dataset_tokenized = tokenize_for_eval(test_dataset, tokenizer)
test_dataset_tokenized

## Load Model

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    trained_weights,
    quantization_config=bnb_config,
    device_map={"": 0}
)

model = PeftModel.from_pretrained(base_model, adapter_path)

## Generate

In [ ]:

def ask_finance_llm(prompt, tokenizer, model):

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


In [ ]:
test_prompt = test_dataset["chat"][0]
ask_finance_llm(test_prompt, tokenizer, model)

In [ ]:
test_df.iloc[0]["answer"]

In [ ]:
from alpha_crunch.finance_llm import format_row

In [ ]:
from alpha_crunch.finance_llm import format_row

# Generate on a subset (full test set takes time — start with 50)
sample_df = test_df.sample(n=50, random_state=42).reset_index(drop=True)

predictions = []
for _, row in sample_df.iterrows():
    prompt = format_row(row, tokenizer, add_answer=False)  # inference mode
    pred   = ask_finance_llm(prompt, model, tokenizer)
    predictions.append(pred)

references = sample_df["answer"].tolist()

## Bertscore

In [ ]:
import bert_score

P, R, F1 = bert_score(predictions, references, lang="en", device="cuda")
print(f"BERTScore — P: {P.mean():.4f} | R: {R.mean():.4f} | F1: {F1.mean():.4f}")

## LLM as Judge

In [ ]:
from openai import OpenAI

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def llm_judge(question, context, reference, prediction):
    prompt = f"""You are evaluating a financial AI assistant. Score the answer 1-5 on each dimension.

Question: {question}
Context: {context}
Reference Answer: {reference}
Model Answer: {prediction}

Score:
- accuracy (is it financially correct?)
- reasoning (does it think like a trader/analyst?)
- completeness (does it fully answer the question?)

Return ONLY valid JSON: {{"accuracy": X, "reasoning": X, "completeness": X, "overall": X}}"""

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    import json
    return json.loads(response.choices[0].message.content)

judge_scores = []
for i, row in sample_df.iterrows():
    scores = llm_judge(
        question=row["question"],
        context=row.get("context", ""),
        reference=row["answer"],
        prediction=predictions[i]
    )
    judge_scores.append(scores)

avg_accuracy    = sum(s["accuracy"]    for s in judge_scores) / len(judge_scores)
avg_reasoning   = sum(s["reasoning"]   for s in judge_scores) / len(judge_scores)
avg_completeness= sum(s["completeness"]for s in judge_scores) / len(judge_scores)
avg_overall     = sum(s["overall"]     for s in judge_scores) / len(judge_scores)

print(f"LLM Judge — Accuracy: {avg_accuracy:.2f} | Reasoning: {avg_reasoning:.2f} | "
      f"Completeness: {avg_completeness:.2f} | Overall: {avg_overall:.2f}")

| Metric                            | Where                       | What It Shows                    |
| --------------------------------- | --------------------------- | -------------------------------- |
| train/loss + eval/loss            | Auto-logged during training | Overfitting detection            |
| test/eval_loss + test/perplexity  | End of train.py             | Baseline quality score           |
| generation_eval/bertscore_f1      | eval.ipynb                  | Semantic answer quality          |
| generation_eval/judge_overall     | eval.ipynb                  | Financial reasoning quality      |
| generation_eval/predictions_table | eval.ipynb                  | Human-readable answer comparison |